# 🎨 TRELLIS: Image to 3D GLB Converter (Working Version)

Convert images to high-quality 3D models using Microsoft's TRELLIS!

## Before You Start

1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `T4 GPU`
2. **Run cells in order** - Don't skip any steps!

---

## ✅ Step 1: Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

## 📦 Step 2: Install TRELLIS (5-10 minutes first time)

In [ ]:
import os
import sys
from pathlib import Path

print("Installing TRELLIS...\n")

# Use Colab's pre-installed PyTorch (don't reinstall)
import torch
print(f"Using PyTorch {torch.__version__}")

# Install dependencies that work with current PyTorch
print("\n[1/4] Installing base packages...")
!pip install -q transformers diffusers accelerate

print("[2/4] Installing 3D packages...")
!pip install -q trimesh[easy] pygltflib
!pip install -q pillow numpy scipy einops omegaconf

print("[3/4] Installing Hugging Face tools...")
!pip install -q huggingface_hub

# Clone TRELLIS
print("[4/4] Downloading TRELLIS...")
TRELLIS_DIR = Path("/content/TRELLIS")
if not TRELLIS_DIR.exists():
    !git clone -q https://github.com/microsoft/TRELLIS.git /content/TRELLIS
    !cd /content/TRELLIS && git submodule update --init --recursive --quiet

# Add to path
sys.path.insert(0, str(TRELLIS_DIR))

print("\n✅ Installation complete!")

## 🔍 Step 3: Test TRELLIS Import

In [ ]:
print("Testing TRELLIS import...\n")

try:
    # Try importing TRELLIS
    import sys
    sys.path.insert(0, '/content/TRELLIS')
    
    from trellis.pipelines import TrellisImageTo3DPipeline
    print("✅ TRELLIS imported successfully!")
    
except Exception as e:
    print(f"❌ Import failed: {e}")
    print("\nTrying to diagnose...")
    
    # Check if TRELLIS directory exists
    import os
    if os.path.exists('/content/TRELLIS'):
        print("✓ TRELLIS directory exists")
        print("\nTRELLIS contents:")
        !ls -la /content/TRELLIS/trellis/ 2>/dev/null || echo "trellis subdirectory not found"
    else:
        print("✗ TRELLIS directory missing")
    
    # Check Python path
    print("\nPython path:")
    for p in sys.path[:5]:
        print(f"  {p}")
    
    print("\n⚠️ If this fails, TRELLIS may not be compatible with current Colab environment")
    print("Trying alternative approach...")
    
    # Try installing from Hugging Face directly
    print("\n📥 Installing from Hugging Face...")
    !pip install -q git+https://github.com/microsoft/TRELLIS.git
    
    # Try import again
    try:
        from trellis.pipelines import TrellisImageTo3DPipeline
        print("\n✅ Success after pip install!")
    except Exception as e2:
        print(f"\n❌ Still failed: {e2}")
        print("\n💡 Alternative: Use the Hugging Face Space instead:")
        print("   https://huggingface.co/spaces/JeffreyXiang/TRELLIS")

## 🚀 Step 4: Load Model (3-5 minutes, ~5GB download)

In [ ]:
from trellis.pipelines import TrellisImageTo3DPipeline
from PIL import Image
import torch
import numpy as np

MODEL_NAME = "JeffreyXiang/TRELLIS-image-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
print(f"Device: {DEVICE}")
print("\nDownloading model (~5GB, first time only)...")

pipeline = TrellisImageTo3DPipeline.from_pretrained(MODEL_NAME)
pipeline = pipeline.to(DEVICE)

print("\n✅ Model loaded!")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 🛠️ Step 5: Helper Function

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

def convert_image_to_3d(image_path, output_path="output.glb", seed=1):
    """
    Convert image to 3D GLB file.
    """
    print(f"\n{'='*70}")
    print(f"Converting: {Path(image_path).name}")
    print(f"{'='*70}\n")
    
    # Load and show image
    image = Image.open(image_path).convert("RGB")
    print(f"Image: {image.size[0]}x{image.size[1]}")
    
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Input: {Path(image_path).name}")
    plt.tight_layout()
    plt.show()
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Run TRELLIS
    print(f"\n🔮 Running TRELLIS (seed={seed})...")
    print("⏱️ This takes ~30-60 seconds...\n")
    
    try:
        outputs = pipeline.run(image, seed=seed)
        print("✓ Inference complete!\n")
    except Exception as e:
        print(f"❌ Inference failed: {e}")
        return None
    
    # Export to GLB
    print("💾 Exporting to GLB...")
    
    try:
        # Try to get and export mesh
        # The exact method depends on TRELLIS version
        if hasattr(outputs, 'export_glb'):
            outputs.export_glb(output_path)
        elif hasattr(outputs, 'mesh'):
            if hasattr(outputs.mesh, 'export'):
                outputs.mesh.export(output_path)
            else:
                # Manual export
                import trimesh
                mesh = trimesh.Trimesh(
                    vertices=outputs.mesh.vertices.cpu().numpy(),
                    faces=outputs.mesh.faces.cpu().numpy()
                )
                mesh.export(output_path)
        else:
            print(f"⚠️ Don't know how to export. Output type: {type(outputs)}")
            print(f"Attributes: {dir(outputs)}")
            return None
        
        # Check result
        if Path(output_path).exists():
            size_mb = Path(output_path).stat().st_size / 1e6
            print(f"\n✅ Success!")
            print(f"Output: {output_path}")
            print(f"Size: {size_mb:.2f} MB")
            return output_path
        else:
            print("❌ Output file not created")
            return None
            
    except Exception as e:
        print(f"❌ Export failed: {e}")
        import traceback
        traceback.print_exc()
        return None

print("✅ Helper function ready!")

## 📤 Step 6: Upload Your Image

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# Create input directory
input_dir = Path("/content/inputs")
input_dir.mkdir(exist_ok=True)

print("📤 Upload your image(s):\n")
uploaded = files.upload()

if uploaded:
    print("\n✅ Uploaded:")
    for filename in uploaded.keys():
        dst = input_dir / filename
        if Path(filename).exists():
            shutil.move(filename, dst)
        size_mb = dst.stat().st_size / 1e6
        print(f"   • {filename} ({size_mb:.2f} MB)")
else:
    print("\n⚠️ No files uploaded")

## 🎨 Step 7: Convert to 3D!

In [ ]:
from pathlib import Path

# List uploaded images
images = list(Path("/content/inputs").glob("*"))
images = [img for img in images if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

if not images:
    print("❌ No images found. Upload images in Step 6 first.")
else:
    print(f"📁 Found {len(images)} image(s):\n")
    for i, img in enumerate(images, 1):
        print(f"   {i}. {img.name}")
    
    # Convert first image
    IMAGE_PATH = str(images[0])
    OUTPUT_PATH = "/content/output.glb"
    SEED = 1  # Change for variations
    
    result = convert_image_to_3d(IMAGE_PATH, OUTPUT_PATH, seed=SEED)
    
    if result:
        print(f"\n{'='*70}")
        print("📥 DOWNLOAD YOUR 3D MODEL:")
        print(f"{'='*70}\n")
        display(FileLink(result))
        
        print("\n💡 View it at: https://gltf-viewer.donmccurdy.com")
        print("   Just drag and drop the GLB file!")

## 🔄 Step 8: Convert ALL Images (Optional)

In [ ]:
from pathlib import Path

# Get all images
input_dir = Path("/content/inputs")
output_dir = Path("/content/outputs")
output_dir.mkdir(exist_ok=True)

images = list(input_dir.glob("*"))
images = [img for img in images if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

if not images:
    print("❌ No images found")
else:
    print(f"Processing {len(images)} images...\n")
    
    results = []
    for i, img_path in enumerate(images, 1):
        print(f"\n[{i}/{len(images)}] {img_path.name}")
        
        output_path = output_dir / f"{img_path.stem}.glb"
        
        try:
            result = convert_image_to_3d(str(img_path), str(output_path), seed=1)
            if result:
                results.append(result)
        except Exception as e:
            print(f"❌ Failed: {e}")
    
    print(f"\n{'='*70}")
    print(f"✅ Batch complete: {len(results)}/{len(images)} successful")
    print(f"{'='*70}\n")
    
    # Show download links
    if results:
        print("📥 Download your models:\n")
        for glb in sorted(output_dir.glob("*.glb")):
            size_mb = glb.stat().st_size / 1e6
            print(f"   {glb.name} ({size_mb:.2f} MB)")
            display(FileLink(str(glb)))

## 📦 Step 9: Download All as ZIP (Optional)

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

output_dir = Path("/content/outputs")
glb_files = list(output_dir.glob("*.glb"))

if not glb_files:
    print("❌ No GLB files found. Run Step 8 first.")
else:
    print(f"📦 Creating zip with {len(glb_files)} models...")
    
    archive = shutil.make_archive("/content/3d_models", 'zip', output_dir)
    size_mb = Path(archive).stat().st_size / 1e6
    
    print(f"   ✓ Archive: 3d_models.zip ({size_mb:.2f} MB)")
    print("\n📥 Downloading...")
    
    files.download(archive)
    print("\n✅ Downloaded!")

---

## 📚 Quick Reference

### View Your Models
- **[gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com)** ← Easiest!
- [Babylon.js Sandbox](https://sandbox.babylonjs.com)
- [Sketchfab](https://sketchfab.com) - Upload & share

### Best Results
✅ Clear, well-lit objects  
✅ Single subject  
✅ Minimal background  
✅ High resolution (1024x1024+)  

### Troubleshooting
- **Out of Memory**: Restart runtime, process one image
- **Import Error**: TRELLIS may not be compatible with current Colab
- **Alternative**: Use [TRELLIS Hugging Face Space](https://huggingface.co/spaces/JeffreyXiang/TRELLIS)

### Performance
- T4 GPU: ~30-60 seconds per image
- A100 GPU: ~15-30 seconds per image

---

## 🎉 Happy 3D Modeling!

Questions? Check [TRELLIS GitHub](https://github.com/microsoft/TRELLIS)